# Preparing Training Data

First of all, we need to convert the training data from .json to .spacy (so that we can train our model)

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer

prefix_re = re.compile(r'[(\[\'"]|.*<i>|.*<sub>|.*<sup>')
suffix_re = re.compile(r'[.,;:?!)\]\'"]|</i>.*|</sub>.*|</sup>.*')
infix_re = re.compile(r'[-/+=&]')

def custom_tokenizer(nlp):
    return Tokenizer(
        nlp.vocab,
        prefix_search=prefix_re.search,
        suffix_search=suffix_re.search,
        infix_finditer=infix_re.finditer
    )

In [ ]:
def print_doc_tokens(title, doc, expected_text, label, start, end): # for debugging
    print('-'*60)
    print(title)
    
    tokens_span = []
    for token in doc:
        tokens_span.append((token, token.idx, token.idx + len(token.text)))
    
    print('Tokens -> ', tokens_span)
    print(f'Expected Entity: {expected_text} (Label: {label}, Start: {start}, End: {end})')

In [ ]:
def prepare_data(json_files):
    db = DocBin()
    for json_file in json_files:
        num_parsed_entities = 0
        num_entities = 0
        print('Parsing {}'.format(json_file))
        print('#'*60)
        f = open(json_file)
        data = json.load(f)
    
        for article in data:
            title = data[article]['metadata']['title']
            abstract = data[article]['metadata']['abstract']
            entities = data[article]['entities']
    
            title_doc = nlp(title)
            abstract_doc = nlp(abstract)
    
            title_ents = []
            abstract_ents = []
            for entity in entities:
                num_entities += 1
                start = int(entity['start_idx'])
                end = int(entity['end_idx']) + 1
                label = entity['label']
                
                if entity['location'] == 'title':
                    span = title_doc.char_span(start, end, label=label, alignment_mode='contract')
                    if span:
                        title_ents.append(span)
                        num_parsed_entities += 1
                    else:
                        # try with expand
                        span = abstract_doc.char_span(start, end, label=label, alignment_mode='expand')
                        if span:
                            abstract_ents.append(span)
                            num_parsed_entities += 1
                        else:
                            print_doc_tokens(title, title_doc, title[start:end], label, start, end)
                elif entity['location'] == 'abstract':
                    span = abstract_doc.char_span(start, end, label=label, alignment_mode='contract')
                    if span:
                        abstract_ents.append(span)
                        num_parsed_entities += 1
                    else:
                        # try with expand
                        span = abstract_doc.char_span(start, end, label=label, alignment_mode='expand')
                        if span:
                            abstract_ents.append(span)
                            num_parsed_entities += 1
                        else:
                            print_doc_tokens(title, abstract_doc, abstract[start:end], label, start, end)
                else:
                    print('ERROR: {}'.format(entity['location']))
    
            title_doc.ents = title_ents

            # If two entities overlap, keep the one that covers more text
            filtered_ents = []
            for ent in sorted(abstract_ents, key=lambda e: (e.start, -len(e.text))):  # Sort by start index, prefer longer entities
                if not any(ent.start < e.end and ent.end > e.start for e in filtered_ents):
                    filtered_ents.append(ent)

            abstract_doc.ents = filtered_ents
            db.add(title_doc)
            db.add(abstract_doc)
        print(f'{num_parsed_entities}/{num_entities}')
        print(f'{len(data)} articles')
    return db

In [ ]:
import json
from spacy.tokens import DocBin

train_json = [
    'gutbrainie2025/Annotations/Train/platinum_quality/json_format/train_platinum.json',
    'gutbrainie2025/Annotations/Train/gold_quality/json_format/train_gold.json',
    'gutbrainie2025/Annotations/Train/silver_quality/json_format/train_silver.json',
    'gutbrainie2025/Annotations/Train/bronze_quality/json_format/train_bronze.json'
]

dev_json = [
    'gutbrainie2025/Annotations/Dev/json_format/dev.json'
]

nlp = spacy.blank('en') # try en_ner_bionlp13cg_md
nlp.tokenizer = custom_tokenizer(nlp)

train_db = prepare_data(train_json)
dev_db = prepare_data(dev_json)
train_db.to_disk('./train.spacy')
dev_db.to_disk('./dev.spacy')

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg
!python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

In [ ]:
!python -m spacy evaluate output/model-best/ dev.spacy

In [ ]:
!python -m spacy evaluate output_tokenizer/model-best/ dev.spacy